# 🧩 04 – Feature Engineering

In this notebook, we create new features to improve the machine learning model's 
ability to detect fraudulent transactions. Feature engineering is often the most 
powerful step in the entire ML pipeline.

In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cleaned_data.csv")
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [12]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

## ⏱️ Time-Based Features
We extract hour, day, month, weekday, and weekend indicators, which often correlate 
with fraud patterns.

In [13]:
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['month'] = df['trans_date_trans_time'].dt.month
df['weekday'] = df['trans_date_trans_time'].dt.weekday
df['is_weekend'] = (df['weekday'] >= 5).astype(int)

## 👤 Age Feature
Older vs younger customers often have different transaction behavior.

In [14]:
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['age'] = (pd.Timestamp('now') - df['dob']).dt.days // 365

## 📍 Distance Between Customer & Merchant (Haversine)
Fraud often involves transactions occurring unusually far from the customer's home.

In [15]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

df['distance_km'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

## 🛒 Encoding Categorical Variables

In [16]:
df = pd.get_dummies(df, columns=['category'], drop_first=True)

In [17]:
df = df.drop(columns=[
    'first', 'last', 'street', 'city', 'state',
    'trans_num', 'job', 'trans_date_trans_time'
])

## 💾 Saving Final Engineered Dataset

In [18]:
df.to_csv("../data/engineered_data.csv", index=False)

In [19]:
df.head()
df.shape

(1852394, 33)